In [ ]:
#!pip install --upgrade transformers

In [9]:
from transformers import pipeline
import json
from pathlib import Path



In [10]:
import transformers
print(transformers.__version__)

5.4.0


In [11]:
# ── 1. Load model ─────────────────────────────────────────────────────────────
# Good small options to try:
# - "mistralai/Mistral-7B-Instruct-v0.3"
# - "Qwen/Qwen2.5-7B-Instruct"
# - "BioMistral/BioMistral-7B" (medical-focused)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

pipe = pipeline(
    "text-generation",
    model=MODEL_NAME,
    device_map="auto",      # automatically uses GPU if available, else CPU
    max_new_tokens=1024,
)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [12]:

# ── 2. Prompt ─────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a medical information extraction assistant.
Given a medical text, extract structured information and return it as JSON only,
with no preamble or explanation.
"""

def build_prompt(text: str) -> str:
    return f"""Extract the following fields from the medical text below.
If a field is not mentioned, use null.

Fields to extract:
- symptoms: list of symptoms reported by the patient
- diagnosis: the diagnosis if mentioned
- medications: list of medications mentioned
- patient_age: age of the patient if mentioned
- patient_gender: gender of the patient if mentioned

Return only a JSON object with these fields.

Medical text:
{text}
"""


In [13]:

# ── 3. Extraction function ────────────────────────────────────────────────────
def extract_medical_info(text: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_prompt(text)}
    ]
    output = pipe(messages)
    raw = output[0]["generated_text"][-1]["content"].strip()

    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        # Return raw text if JSON parsing fails, so you can inspect it
        print(f"Warning: could not parse JSON, returning raw output.")
        return {"raw_output": raw}



In [15]:
# ── 4. Process a folder of .txt files ────────────────────────────────────────
def process_folder(folder_path: str) -> list[dict]:
    results = []
    for txt_file in Path(folder_path).glob("*.txt"):
        text = txt_file.read_text(encoding="utf-8")
        print(f"Processing {txt_file.name}...")
        extracted = extract_medical_info(text)
        extracted["source_file"] = txt_file.name
        results.append(extracted)
    return results


In [16]:

# ── 5. Example usage ──────────────────────────────────────────────────────────

sample = """
    John, a 45-year-old male, presented with severe headache, fever of 39°C,
    and stiff neck for the past 2 days. He was prescribed ibuprofen and referred
    for a lumbar puncture to rule out meningitis.
    """
result = extract_medical_info(sample)
print(json.dumps(result, indent=2))

# Folder of files — uncomment to use
# results = process_folder("path/to/your/texts")
# import pandas as pd
# df = pd.DataFrame(results)
# df.to_csv("extracted_medical_info.csv", index=False)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "symptoms": [
    "severe headache",
    "fever (39\u00b0C)",
    "stiff neck"
  ],
  "diagnosis": "meningitis",
  "medications": [
    "ibuprofen"
  ],
  "patient_age": "45",
  "patient_gender": "male"
}


In [8]:
!du -sh ~/.cache/huggingface/hub/

256G	/home/tatiana.bladier/.cache/huggingface/hub/
